# Continuous bandwidth minimizers: ISE, Q₀, Q₁, and Q₁₃/₁₄

For each $n=100,200,\ldots,2000$, compare the sampling distributions of four continuously optimized bandwidths on the common closed interval $h\in[0.01,2]$:

| Selector | Sample criterion |
|---|---|
| ISE | $\int(\widehat f_h-f)^2$ against the known $f=N(0,1)$ |
| $\widehat Q_0$ | $\tfrac12h^2\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_1$ | $\tfrac12\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_{13/14}$ | $\tfrac12h^{1/7}\|\partial_h\widehat f_h\|_2^2$ |

These subscripts use the thesis's **r parameter**, with
$\widehat Q_r(h)=\tfrac12h^{2(1-r)}\|\partial_h\widehat f_h\|_2^2$.
The ISE selector is the sample-specific oracle available here because the simulation density is known. Every criterion is evaluated with analytic Gaussian formulas. The search refines interior candidates continuously and compares both endpoints. Endpoint selections are part of the reported distributions.

The default is **M=1000 paired repetitions at each sample size**. The original 1000-by-1000 sample bank from seed `20260830` is retained exactly. A second block of 1000 observations per repetition is generated independently with `SeedSequence([20260830, 1])` and appended. Thus every sample for n≤1000 is unchanged, and the larger samples extend those same repetitions. Verified checkpoints from the original run are imported for n≤1000. A preview uses the first 12 repetitions of the same complete bank.

Sources: [thesis ISE and `prop:main_thm` defining Q_r](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [exact Gaussian ISE helper](kde_tools/density.py); [continuous selector implementation and derived log-h derivatives](kde_tools/continuous_minimizers.py); [earlier paired-sample settings](outputs/data/fast_l2_bandwidth_distributions_vs_n_lambda/thesis/028fa8664f9c/settings.json).

Run all cells. A separate resumable checkpoint is saved for each n. The figures are saved automatically as PDFs and also as PNGs. Two pages each show the complete interval and the interior selections with separate bars for the probability at the upper endpoint h=2. All histogram frequencies use M as denominator.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

working_directory = Path.cwd().resolve()
code_directory = next(
    (candidate.resolve()
     for root in (working_directory, *working_directory.parents)
     for candidate in (root, root / "Code" / "Code_kde")
     if (candidate / "kde_tools/pdf_export.py").is_file()), None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from the project or Code/Code_kde folder.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(code_directory / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from threadpoolctl import threadpool_limits

from kde_tools.continuous_minimizers import (
    CONTINUOUS_SELECTOR_VERSION, METHODS, R_VALUES, continuous_ise_qr_minima,
)
from kde_tools.density import exact_oracle_ise_values
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = "continuous_ise_qhat_distributions_vs_n"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name, output_root=code_directory / "outputs/pdfs"
)
data_output_directory = code_directory / "outputs/data" / notebook_name
image_output_directory = code_directory / "outputs/images" / notebook_name
print(f"Automatic PDF exports: {pdf_output_directory}")


## Settings and paired Gaussian samples

The optimization interval is the same [0.01,2] used in the earlier experiment. `HISTOGRAM_POINTS` controls the common display bins only. The minimizers are continuously refined and can fall between bin centers. The three Q criteria are sample energies, optimized separately within each repetition.


In [ ]:
RUN_PROFILE = "thesis"  # "thesis" or "preview"
PROFILES = {"thesis": {"repetitions": 1000}, "preview": {"repetitions": 12}}
SAMPLE_SIZES = list(range(100, 2001, 100))
SEED = 20260830
SAMPLE_BANK_REPETITIONS = 1000
SAMPLE_BANK_MAXIMUM_N = 2000
SAMPLE_BANK_BLOCK_SIZE = 1000
PREFIX_RUN_ID = "b4afbc6cd382"
H_MIN, H_MAX = 0.01, 2.0
SIGMA2 = 1.0
SCAN_POINTS = 257
SPATIAL_POINTS = 32768
HALF_WIDTH = 16.0
HISTOGRAM_POINTS = 201
WORKERS = 6
REUSE_CHECKPOINTS = True

M = PROFILES[RUN_PROFILE]["repetitions"]
if not 1 <= M <= SAMPLE_BANK_REPETITIONS:
    raise ValueError("Repetitions must fit in the fixed paired sample bank.")
if not SAMPLE_SIZES or min(SAMPLE_SIZES) < 1 or max(SAMPLE_SIZES) > SAMPLE_BANK_MAXIMUM_N:
    raise ValueError("Sample sizes must fit in the fixed sample bank.")
sample_blocks = []
for block_index in range((SAMPLE_BANK_MAXIMUM_N + SAMPLE_BANK_BLOCK_SIZE - 1) // SAMPLE_BANK_BLOCK_SIZE):
    block_seed = SEED if block_index == 0 else np.random.SeedSequence([SEED, block_index])
    sample_blocks.append(np.random.default_rng(block_seed).normal(
        0.0, np.sqrt(SIGMA2), size=(SAMPLE_BANK_REPETITIONS, SAMPLE_BANK_BLOCK_SIZE)
    ))
full_sample_bank = np.concatenate(sample_blocks, axis=1)[:, :SAMPLE_BANK_MAXIMUM_N]
prefix_sample_bank_sha256 = hashlib.sha256(full_sample_bank[:, :SAMPLE_BANK_BLOCK_SIZE].tobytes()).hexdigest()
full_samples = full_sample_bank[:M]
configuration = {
    "version": CONTINUOUS_SELECTOR_VERSION, "methods": list(METHODS),
    "r_values": R_VALUES, "sample_sizes": SAMPLE_SIZES, "repetitions": M,
    "seed": SEED, "sigma2": SIGMA2, "h_min": H_MIN, "h_max": H_MAX,
    "sample_bank_shape": list(full_sample_bank.shape),
    "sample_bank_generation": "fixed_1000_column_blocks_preserving_original_prefix_v1",
    "sample_bank_block_size": SAMPLE_BANK_BLOCK_SIZE,
    "prefix_sample_bank_sha256": prefix_sample_bank_sha256,
    "prefix_run_id": PREFIX_RUN_ID,
    "sample_bank_sha256": hashlib.sha256(full_sample_bank.tobytes()).hexdigest(),
    "scan_points": SCAN_POINTS, "spatial_points": SPATIAL_POINTS,
    "half_width": HALF_WIDTH, "histogram_points": HISTOGRAM_POINTS,
    "source_sha256": {
        filename: hashlib.sha256((code_directory / "kde_tools" / filename).read_bytes()).hexdigest()
        for filename in ("continuous_minimizers.py", "qhat.py")
    },
}
configuration_json = json.dumps(configuration, sort_keys=True)
configuration_id = hashlib.sha256(configuration_json.encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
image_directory = image_output_directory / RUN_PROFILE / configuration_id
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(run_directory / "settings.json").write_text(json.dumps(configuration, indent=2) + "\n")
histogram_bandwidths = np.geomspace(H_MIN, H_MAX, HISTOGRAM_POINTS)
histogram_positions = np.arange(HISTOGRAM_POINTS, dtype=float)
histogram_edges = np.arange(HISTOGRAM_POINTS + 1) - 0.5
prefix_run_directory = data_output_directory / "thesis" / PREFIX_RUN_ID
prefix_configuration = None
if (prefix_run_directory / "settings.json").exists():
    prefix_configuration = json.loads((prefix_run_directory / "settings.json").read_text())
    keys = ("version", "methods", "r_values", "seed", "sigma2", "h_min", "h_max",
            "scan_points", "spatial_points", "half_width", "source_sha256")
    if (any(prefix_configuration[key] != configuration[key] for key in keys)
            or prefix_configuration["sample_bank_sha256"] != prefix_sample_bank_sha256
            or prefix_configuration["repetitions"] < M):
        raise ValueError("Original checkpoints cannot be imported: samples or objectives differ.")
print(f"M={M} paired repetitions; n={SAMPLE_SIZES}; continuous interval=[{H_MIN},{H_MAX}]")
print(f"Checkpoint and CSV directory: {run_directory}")


## Compute or resume all four continuous minimizers

The same sample supplies all four selectors. A Fourier scan supplies candidate basins, followed by analytic Gaussian-pair refinement and comparison with both endpoints. A finite scan is a numerical search, rather than a certificate that every stationary point was enumerated. Later validation doubles both scan and Fourier resolution on three repetitions at every n.


In [ ]:
FIELDS = ("h", "objective", "log_derivative", "log_curvature", "boundary", "interior_candidates")


def select_one_sample(sample):
    return continuous_ise_qr_minima(
        sample, H_MIN, H_MAX, sigma2=SIGMA2, scan_points=SCAN_POINTS,
        spatial_points=SPATIAL_POINTS, half_width=HALF_WIDTH,
    )


def run_sample_size(n, *, reuse=True):
    checkpoint_path = run_directory / f"n{n}.npz"
    values = np.full((M, len(METHODS), len(FIELDS)), np.nan)
    completed = 0
    imported_from = None
    if reuse and checkpoint_path.exists():
        with np.load(checkpoint_path, allow_pickle=False) as saved:
            if str(saved["configuration_json"].item()) != configuration_json or int(saved["n"]) != n:
                raise ValueError("Checkpoint configuration or sample size differs.")
            if list(saved["methods"]) != list(METHODS) or list(saved["fields"]) != list(FIELDS):
                raise ValueError("Checkpoint selector or field ordering differs.")
            values = saved["values"].copy()
            completed = int(saved["completed"])
    elif (reuse and prefix_configuration is not None and n <= SAMPLE_BANK_BLOCK_SIZE
          and (prefix_run_directory / f"n{n}.npz").exists()):
        with np.load(prefix_run_directory / f"n{n}.npz", allow_pickle=False) as saved:
            if (str(saved["configuration_json"].item()) != json.dumps(prefix_configuration, sort_keys=True)
                    or int(saved["n"]) != n or list(saved["methods"]) != list(METHODS)
                    or list(saved["fields"]) != list(FIELDS)):
                raise ValueError("Original checkpoint metadata differs from its validated settings.")
            if int(saved["completed"]) >= M:
                values = saved["values"][:M].copy()
                completed = M
                imported_from = str(prefix_run_directory / f"n{n}.npz")
    if values.shape != (M, len(METHODS), len(FIELDS)) or not 0 <= completed <= M:
        raise ValueError("Invalid checkpoint shape or completion count.")
    if not np.all(np.isfinite(values[:completed])):
        raise ValueError("Completed checkpoint rows must be finite.")

    def save():
        temporary = checkpoint_path.with_suffix(".npz.tmp")
        with temporary.open("wb") as stream:
            np.savez_compressed(
                stream, configuration_json=configuration_json, n=n, values=values,
                completed=completed, methods=np.array(METHODS), fields=np.array(FIELDS),
                imported_from=np.array(imported_from or ""),
            )
        temporary.replace(checkpoint_path)

    if imported_from is not None:
        save()
        print(f"n={n}: imported {completed} unchanged paired samples from the original run.", flush=True)
    print(f"n={n}: starting at {completed}/{M}", flush=True)
    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        pending = (full_samples[r, :n] for r in range(completed, M))
        for result in pool.map(select_one_sample, pending):
            for method_index, method in enumerate(METHODS):
                values[completed, method_index] = [result[method][field] for field in FIELDS]
            completed += 1
            if completed % 25 == 0 or completed == M:
                save()
            if completed % 250 == 0 or completed == M:
                print(f"n={n}: {completed}/{M}; {time.perf_counter()-started:.1f}s", flush=True)
    if np.any(values[:, :, 0] < H_MIN) or np.any(values[:, :, 0] > H_MAX):
        raise ArithmeticError("Continuous minimizer outside the shared interval.")
    return values


results_by_n = {}
with threadpool_limits(limits=1):
    select_one_sample(full_samples[0, :SAMPLE_SIZES[0]])  # Warm optional compiled pair sums.
    for current_n in SAMPLE_SIZES:
        results_by_n[current_n] = run_sample_size(current_n, reuse=REUSE_CHECKPOINTS)


## Saved distributions, boundary counts, and summaries

Each CSV selection row records a continuously optimized bandwidth and its numerical diagnostics. Histogram frequencies use the same geometric bin centers for all four selectors and divide by M. The summary explicitly counts selections at both endpoints. Interior-only frequencies below keep the same denominator M.

`peak_comparison.csv` records the ISE mode bin, the green interior mode bin, boundary mass, and paired bandwidth comparisons on samples where Q₁₃/₁₄ has an interior minimizer. These are descriptive histogram and sample comparisons.


In [ ]:
frequencies, interior_frequencies, boundary_masses = {}, {}, {}
selection_rows, frequency_rows, summary_rows = [], [], []
for current_n in SAMPLE_SIZES:
    values = results_by_n[current_n]
    frequencies[current_n], interior_frequencies[current_n], boundary_masses[current_n] = {}, {}, {}
    for method_index, method in enumerate(METHODS):
        selected = values[:, method_index, 0]
        boundary = values[:, method_index, 4].astype(int)
        positions = np.interp(np.log(selected), np.log(histogram_bandwidths), histogram_positions)
        counts = np.histogram(positions, bins=histogram_edges)[0]
        interior_counts = np.histogram(positions[boundary != 1], bins=histogram_edges)[0]
        frequencies[current_n][method] = counts / M
        interior_frequencies[current_n][method] = interior_counts / M
        boundary_masses[current_n][method] = float(np.mean(boundary == 1))
        assert counts.sum() == M
        assert interior_counts.sum() + np.sum(boundary == 1) == M
        for repetition in range(M):
            selection_rows.append({
                "n": current_n, "repetition": repetition, "method": method,
                **{field: float(values[repetition, method_index, i]) for i, field in enumerate(FIELDS)},
            })
        for index in range(HISTOGRAM_POINTS):
            frequency_rows.append({
                "n": current_n, "method": method, "grid_index": index,
                "h": float(histogram_bandwidths[index]), "count": int(counts[index]),
                "relative_frequency": float(counts[index] / M),
                "interior_relative_frequency": float(interior_counts[index] / M),
            })
        summary_rows.append({
            "n": current_n, "method": method, "M": M,
            "mean_h": float(np.mean(selected)), "median_h": float(np.median(selected)),
            "sd_h": float(np.std(selected, ddof=1)),
            "q05_h": float(np.quantile(selected, .05)), "q95_h": float(np.quantile(selected, .95)),
            "lower_boundary_count": int(np.sum(boundary == -1)),
            "upper_boundary_count": int(np.sum(boundary == 1)),
            "upper_boundary_frequency": boundary_masses[current_n][method],
            "interior_count": int(np.sum(boundary == 0)),
            "max_interior_candidates": int(np.max(values[:, method_index, 5])),
        })
for filename, rows in (("continuous_minimizers.csv", selection_rows),
                       ("frequencies.csv", frequency_rows), ("summary.csv", summary_rows)):
    with (run_directory / filename).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
print(" n    selector      median h    at h=2")
for row in summary_rows:
    print(f"{row['n']:4d}  {row['method']:9s}  {row['median_h']:10.5f}  {row['upper_boundary_frequency']:8.1%}")

# Compare the green interior mode with the ISE mode on the common display bins.
peak_comparison_rows = []
for current_n in SAMPLE_SIZES:
    values = results_by_n[current_n]
    ise_mode = float(histogram_bandwidths[np.argmax(frequencies[current_n]["ISE"])])
    green_interior = values[:, METHODS.index("Q13_14"), 4] == 0
    if np.any(green_interior):
        green_mode = float(histogram_bandwidths[np.argmax(interior_frequencies[current_n]["Q13_14"])])
        paired_ise = values[green_interior, METHODS.index("ISE"), 0]
        paired_green = values[green_interior, METHODS.index("Q13_14"), 0]
        mode_ratio = green_mode / ise_mode
        paired_ratio = float(np.median(paired_green / paired_ise))
        paired_difference = float(np.median(paired_green - paired_ise))
    else:
        green_mode = mode_ratio = paired_ratio = paired_difference = None
    peak_comparison_rows.append({
        "n": current_n, "ise_mode_bin_h": ise_mode,
        "q13_14_interior_mode_bin_h": green_mode,
        "interior_mode_ratio_q13_14_over_ise": mode_ratio,
        "q13_14_interior_count": int(np.sum(green_interior)),
        "q13_14_upper_boundary_frequency": boundary_masses[current_n]["Q13_14"],
        "paired_median_h_ratio_on_q13_14_interior_samples": paired_ratio,
        "paired_median_h_difference_on_q13_14_interior_samples": paired_difference,
    })
with (run_directory / "peak_comparison.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=peak_comparison_rows[0].keys())
    writer.writeheader()
    writer.writerows(peak_comparison_rows)


## Numerical validation

Repeat the continuous searches at doubled scan and Fourier resolution on three paired repetitions at each n. Independently evaluate the exact ISE helper at all four selected bandwidths and both endpoints for these samples; the selected ISE must be no larger than any of those alternatives. Check stationarity and positive curvature for all interior solutions, plus the appropriate one-sided derivative signs at endpoints.


In [ ]:
validation_rows = []
with threadpool_limits(limits=1):
    for current_n in SAMPLE_SIZES:
        values = results_by_n[current_n]
        interior = values[:, :, 4] == 0
        assert np.all(values[:, :, 3][interior] > 0)
        assert np.all(np.abs(values[:, :, 2][interior] / values[:, :, 3][interior]) <= 1e-7)
        assert np.all(values[:, :, 2][values[:, :, 4] == 1] <= 1e-8)
        assert np.all(values[:, :, 2][values[:, :, 4] == -1] >= -1e-8)
        for repetition in sorted({0, M // 2, M - 1}):
            sample = full_samples[repetition, :current_n]
            refined = continuous_ise_qr_minima(
                sample, H_MIN, H_MAX, sigma2=SIGMA2, scan_points=2*SCAN_POINTS-1,
                spatial_points=2*SPATIAL_POINTS, half_width=HALF_WIDTH,
            )
            for method_index, method in enumerate(METHODS):
                difference = abs(refined[method]["h"] - values[repetition, method_index, 0])
                if difference > 1e-7:
                    raise ArithmeticError(f"Refined search changed {method} at n={current_n}.")
                validation_rows.append({"n": current_n, "repetition": repetition, "method": method,
                                        "bandwidth_discrepancy": float(difference)})
            alternative_h = np.r_[values[repetition, :, 0], H_MIN, H_MAX]
            independent_ise = exact_oracle_ise_values(sample, alternative_h, SIGMA2)
            if independent_ise[0] > independent_ise.min() + 1e-10:
                raise ArithmeticError("ISE selector did not minimize the independent candidate scores.")
            if abs(independent_ise[0] - values[repetition, 0, 1]) > 1e-10:
                raise ArithmeticError("Analytic ISE disagrees with the shared exact helper.")
validation_report = {
    "completed_samples": len(SAMPLE_SIZES)*M,
    "continuous_minimizers": len(SAMPLE_SIZES)*M*len(METHODS),
    "maximum_refinement_discrepancy": max(row["bandwidth_discrepancy"] for row in validation_rows),
    "checks": ["interval bounds", "probability totals", "interior stationarity and curvature",
               "one-sided boundary derivative signs", "doubled search resolution",
               "independent exact ISE scores"],
    "refinement_rows": validation_rows,
}
(run_directory / "validation.json").write_text(json.dumps(validation_report, indent=2) + "\n")
print(f"Validated {validation_report['continuous_minimizers']} continuous minimizers.")
print(f"Maximum change after search refinement: {validation_report['maximum_refinement_discrepancy']:.3e}")


## Distribution plots

The first figure shows all selections on the complete bandwidth interval. In the second, each main panel shows the histograms after removing only exact upper-endpoint selections; the inset bars show the probability at h=2. Main-panel frequencies still divide by all M repetitions. The main-panel range includes every occupied interior bin across all sample sizes and methods. Together, the interior histograms and inset bars account for the complete probability distributions.

The twenty sample sizes are split into two readable pages: n=100,…,1000 and n=1100,…,2000. Axis limits are shared across both pages.


In [ ]:
COLORS = {"ISE": "#7B4AB2", "Q0": "#0072B2", "Q1": "#D55E00", "Q13_14": "#009E73"}
LABELS = {"ISE": "ISE", "Q0": r"$\widehat Q_0$", "Q1": r"$\widehat Q_1$",
          "Q13_14": r"$\widehat Q_{13/14}$"}
STYLES = {"ISE": "-", "Q0": "-", "Q1": "--", "Q13_14": "-."}
legend_handles = [Line2D([0], [0], color=COLORS[m], linestyle=STYLES[m], linewidth=2,
                         label=LABELS[m]) for m in METHODS]


def style_panel(ax, current_n, ticks, y_max, right_limit):
    ax.set_title(rf"$n={current_n}$", fontsize=13, pad=8)
    ax.set_xticks(ticks, [f"{histogram_bandwidths[i]:.3g}" for i in ticks])
    ax.set_xlim(-.7, right_limit)
    ax.set_ylim(0, y_max)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    ax.set_xlabel("continuously selected bandwidth h", fontsize=10)
    ax.set_ylabel("relative frequency", fontsize=10)
    ax.tick_params(labelsize=9, colors="#304967", labelbottom=True, labelleft=True)
    ax.grid(axis="y", color="#DDE5EC", linewidth=.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#B8C5D1")


def draw_histograms(ax, distributions):
    for method in METHODS:
        ax.stairs(distributions[method], histogram_edges, fill=True,
                  color=COLORS[method], alpha=.07, linewidth=0)
        ax.stairs(distributions[method], histogram_edges, color=COLORS[method],
                  linestyle=STYLES[method], linewidth=1.8 if method == "ISE" else 1.5)


sample_size_pages = [SAMPLE_SIZES[i:i+10] for i in range(0, len(SAMPLE_SIZES), 10)]
full_y_max = min(1.06, 1.06*max(v.max() for point in frequencies.values() for v in point.values()))
full_ticks = np.unique(np.linspace(0, HISTOGRAM_POINTS-1, 7, dtype=int))
full_png_paths = []
for page in sample_size_pages:
    rows = (len(page)+1)//2
    full_fig, full_axes = plt.subplots(rows, 2, figsize=(14, 3.2*rows+2), sharex=True, sharey=True, squeeze=False)
    for ax, current_n in zip(full_axes.flat, page):
        draw_histograms(ax, frequencies[current_n])
        style_panel(ax, current_n, full_ticks, full_y_max, HISTOGRAM_POINTS-.3)
    for ax in list(full_axes.flat)[len(page):]:
        ax.set_visible(False)
    full_fig.suptitle(
        f"Continuous minimizers, n={page[0]}–{page[-1]}: full interval\n"
        + rf"Gaussian samples; $M={M}$ per $n$; $h\in[{H_MIN:g},{H_MAX:g}]$",
        fontsize=16, y=.995,
    )
    full_fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(.5,.953),
                    ncol=4, frameon=False, fontsize=12)
    full_fig.tight_layout(rect=(0,0,1,.925), h_pad=2, w_pad=2)
    full_png_path = image_directory / f"continuous_ise_Q0_Q1_Q13over14_full_n{page[0]}_{page[-1]}_M{M}.png"
    full_fig.savefig(full_png_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(full_fig)
    full_png_paths.append(full_png_path)

occupied = np.flatnonzero(np.any(np.stack([
    v > 0 for point in interior_frequencies.values() for v in point.values()
]), axis=0))
interior_last_index = min(HISTOGRAM_POINTS-1, int(occupied[-1])+3)
interior_ticks = np.unique(np.linspace(0, interior_last_index, 7, dtype=int))
interior_y_max = 1.12*max(v.max() for point in interior_frequencies.values() for v in point.values())
interior_png_paths = []
for page in sample_size_pages:
    rows = (len(page)+1)//2
    interior_fig, interior_axes = plt.subplots(rows, 2, figsize=(14,3.2*rows+2), sharex=True, sharey=True, squeeze=False)
    for ax, current_n in zip(interior_axes.flat, page):
        draw_histograms(ax, interior_frequencies[current_n])
        style_panel(ax, current_n, interior_ticks, interior_y_max, interior_last_index+.5)
        boundary_ax = ax.inset_axes([.14,.35,.23,.48])
        boundary_ax.bar(np.arange(len(METHODS)), [boundary_masses[current_n][m] for m in METHODS],
                        color=[COLORS[m] for m in METHODS], width=.67)
        boundary_ax.set_ylim(0,1.05)
        boundary_ax.set_yticks([0,.5,1], ["0%","50%","100%"])
        boundary_ax.set_xticks([])
        boundary_ax.tick_params(axis="y", labelsize=7, length=2)
        boundary_ax.set_title(rf"at $h={H_MAX:g}$", fontsize=9, pad=4)
        boundary_ax.patch.set_facecolor("#FAFBFD")
        for side in ("top", "right"):
            boundary_ax.spines[side].set_visible(False)
        ax.text(.02,.95,
                f"At h={H_MAX:g}: " + rf"$Q_1$: {boundary_masses[current_n]['Q1']:.1%}; "
                + rf"$Q_{{13/14}}$: {boundary_masses[current_n]['Q13_14']:.1%}",
                transform=ax.transAxes, va="top", fontsize=8, color="#304967")
    for ax in list(interior_axes.flat)[len(page):]:
        ax.set_visible(False)
    interior_fig.suptitle(
        f"Continuous minimizers, n={page[0]}–{page[-1]}: interior and boundary mass\n"
        + rf"Gaussian samples; $M={M}$ per $n$; all frequencies divide by $M$",
        fontsize=16, y=.995,
    )
    interior_fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(.5,.953),
                        ncol=4, frameon=False, fontsize=12)
    interior_fig.tight_layout(rect=(0,0,1,.925), h_pad=2, w_pad=2)
    interior_png_path = image_directory / f"continuous_ise_Q0_Q1_Q13over14_interior_and_boundary_n{page[0]}_{page[-1]}_M{M}.png"
    interior_fig.savefig(interior_png_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(interior_fig)
    interior_png_paths.append(interior_png_path)
for path in full_png_paths + interior_png_paths:
    print(f"Saved distribution plot: {path}")
print(f"Automatic PDF directory: {pdf_output_directory}")
